In [ ]:
import numpy as np
import torch
import time
from scipy import sparse as sp
from scipy.linalg import eigh
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

pauli_matrices = {
    1: np.array([[0, 1], [1, 0]], dtype=complex),
    2: np.array([[0, -1j], [1j, 0]], dtype=complex),
    3: np.array([[1, 0], [0, -1]], dtype=complex),
    0: np.eye(2, dtype=complex)
}
pauli_matrices_sparse = {key: sp.csr_matrix(value) for key, value in pauli_matrices.items()}

def pauli_site(x, i, L):
    """Pauli matrix on site i (sparse)"""
    if L == 1:
        return pauli_matrices_sparse[x]
    if i == 1:
        return sp.kron(pauli_matrices_sparse[x], sp.eye(2**(L-1), format='csr'))
    else:
        return sp.kron(sp.eye(2), pauli_site(x, i-1, L-1), format='csr')

def pauli_int(spin, site, L):
    """Nearest‑neighbour interaction term (periodic)"""
    dim = 2**L
    if site == L:
        op = sp.eye(dim, dtype=complex, format='csr')
        for i in range(1, L+1):
            if i == L or i == 1:
                op = op @ pauli_site(spin, i, L)
        return op
    else:
        op = sp.eye(dim, dtype=complex, format='csr')
        for i in range(1, L+1):
            if i == site or i == site+1:
                op = op @ pauli_site(spin, i, L)
        return op

def xxz_hamiltonian(L, delta=1):
    """Pure XXZ Hamiltonian (nearest neighbour)"""
    dim = 2**L
    H = sp.csr_matrix((dim, dim), dtype=complex)
    for site in range(1, L+1):
        H += pauli_int(1, site, L)
        H += pauli_int(2, site, L)
        H += delta * pauli_int(3, site, L)
    return H

def periodic_field_hamiltonian(L, delta=1, Q=np.pi, h=0.5):
    """XXX + longitudinal periodic field (with an overall sign as in original code)"""
    H0 = -xxz_hamiltonian(L, delta)   
    dim = 2**L
    Hz = sp.csr_matrix((dim, dim), dtype=complex)
    for n in range(1, L+1):
        Hz += np.cos(Q * n) * pauli_site(3, n, L)
    H = H0 + h * Hz
    return -H   

def build_hamiltonian(model, L, **kwargs):
    """Construct Hamiltonian for the chosen model"""
    if model == "XXZ":
        delta = kwargs.get('delta', 1.0)
        return xxz_hamiltonian(L, delta)
    elif model == "PeriodicField":
        delta = kwargs.get('delta', 1.0)
        Q = kwargs.get('Q', np.pi)
        h = kwargs.get('h', 0.5)
        return periodic_field_hamiltonian(L, delta, Q, h)

def local_ops(dtype=torch.complex64, device="cpu"):
    Sz = torch.tensor([[0.5, 0.0], [0.0, -0.5]], dtype=dtype, device=device)
    Sp = torch.tensor([[0.0, 1.0], [0.0, 0.0]], dtype=dtype, device=device)
    Sm = torch.tensor([[0.0, 0.0], [1.0, 0.0]], dtype=dtype, device=device)
    Sx = torch.tensor([[0.0, 1.0], [1.0, 0.0]], dtype=dtype, device=device)
    Sy = torch.tensor([[0.0, -1j], [1j, 0.0]], dtype=dtype, device=device)
    I2 = torch.eye(2, dtype=dtype, device=device)
    return I2, Sx, Sy, Sz, Sp, Sm

def precompute_perms(L: int):
    perms, inv_perms = [], []
    axes = list(range(L))
    for s in range(L):
        p = [s] + [a for a in axes if a != s]
        inv = np.argsort(p)
        perms.append(p)
        inv_perms.append(inv.tolist())
    return perms, inv_perms

def apply_one_site(op2, site, vec, L, perms, inv_perms):
    psi = vec.reshape([2]*L).permute(perms[site]).reshape(2, -1)
    psi2 = op2 @ psi
    out = psi2.reshape([2]*L).permute(inv_perms[site]).reshape(-1)
    return out

def B_action(u, psi, L, I2, Sz, Sp, Sm, perms, inv_perms):
    dtype, device = psi.dtype, psi.device
    ii = torch.tensor(1j, dtype=dtype, device=device)
    V11 = psi.clone()
    V12 = torch.zeros_like(psi)
    V21 = torch.zeros_like(psi)
    V22 = psi.clone()
    for site in range(L):
        SzV11 = apply_one_site(Sz, site, V11, L, perms, inv_perms)
        SzV12 = apply_one_site(Sz, site, V12, L, perms, inv_perms)
        SzV21 = apply_one_site(Sz, site, V21, L, perms, inv_perms)
        SzV22 = apply_one_site(Sz, site, V22, L, perms, inv_perms)
        SmV21 = apply_one_site(Sm, site, V21, L, perms, inv_perms)
        SmV22 = apply_one_site(Sm, site, V22, L, perms, inv_perms)
        SpV11 = apply_one_site(Sp, site, V11, L, perms, inv_perms)
        SpV12 = apply_one_site(Sp, site, V12, L, perms, inv_perms)
        L11V11 = u * V11 + ii * SzV11
        L11V12 = u * V12 + ii * SzV12
        L22V21 = u * V21 - ii * SzV21
        L22V22 = u * V22 - ii * SzV22
        L12V21 = ii * SmV21
        L12V22 = ii * SmV22
        L21V11 = ii * SpV11
        L21V12 = ii * SpV12
        N11 = L11V11 + L12V21
        N12 = L11V12 + L12V22
        N21 = L21V11 + L22V21
        N22 = L21V12 + L22V22
        V11, V12, V21, V22 = N11, N12, N21, N22
    return V12

def vacuum_state(L, dtype=torch.complex64, device="cpu"):
    v = torch.zeros(2**L, dtype=dtype, device=device)
    v[0] = 1.0 + 0.0j
    return v

def bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True):
    psi = vacuum_state(L, dtype=I2.dtype, device=I2.device)
    for layer_params in params_list:
        u = layer_params[0] + 1j * layer_params[1]
        psi = B_action(u, psi, L, I2, Sz, Sp, Sm, perms, inv_perms)
    if normalize:
        norm = torch.sqrt(torch.real(torch.vdot(psi, psi)))
        if norm > 1e-15:
            psi = psi / norm
    return psi

def exact_diagonalization(H, num_states=3):
    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    eigenvalues, eigenvectors = eigh(H_dense)
    results = []
    for i in range(min(num_states, len(eigenvalues))):
        results.append({'energy': eigenvalues[i], 'state': eigenvectors[:, i]})
    return results

def ground_loss_func_degenerate(params, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers):
    params_per_layer = 2
    total_layers = len(params) // params_per_layer
    params_list = []
    for layer in range(total_layers):
        start_idx = layer * params_per_layer
        re_part = params[start_idx]
        im_part = params[start_idx + 1]
        params_list.append([re_part, im_part])
    psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
    H_psi = H_tensor @ psi
    energy = torch.real(torch.vdot(psi, H_psi))
    norm2 = torch.real(torch.vdot(psi, psi))
    return energy / norm2

def torch_wrapper_degenerate(params_numpy, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers):
    params = torch.tensor(params_numpy, dtype=torch.float64, requires_grad=True)
    loss = ground_loss_func_degenerate(params, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers)
    loss.backward()
    loss_value = loss.item()
    grad_value = params.grad.numpy().astype(np.float64)
    return loss_value, grad_value

def optimize_degenerate_ground_state(model, L, num_layers, num_attempts, maxiter, device="cpu", **model_kwargs):
    """
    For models where the ground state is degenerate (e.g., PeriodicField with certain parameters).
    Fidelity is computed as sum of squared overlaps with the first two exact eigenstates.
    """
    print(f"\n=== Degenerate ground state optimization (energy minimization) ===")
    print(f"Model: {model}, L={L}, layers={num_layers}, attempts={num_attempts}")
    H = build_hamiltonian(model, L, **model_kwargs)
    exact_states = exact_diagonalization(H, num_states=3)
    ground_energy_exact = exact_states[0]['energy']
    excited1_energy_exact = exact_states[1]['energy']
    excited2_energy_exact = exact_states[2]['energy']
    print(f"Exact ground energy: {ground_energy_exact:.8f}")
    print(f"Exact first excited: {excited1_energy_exact:.8f}")
    print(f"Exact second excited: {excited2_energy_exact:.8f}")

    I2, Sx, Sy, Sz, Sp, Sm = local_ops(dtype=torch.complex128, device=device)
    perms, inv_perms = precompute_perms(L)
    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    H_tensor = torch.tensor(H_dense, dtype=torch.complex128, device=device)

    exact_states_tensor = [torch.tensor(st['state'], dtype=torch.complex128, device=device) for st in exact_states]

    best_energy = float('inf')
    best_params = None
    best_psi = None
    best_fidelity = 0.0

    for attempt in range(num_attempts):
        print(f"Attempt {attempt+1}/{num_attempts}:")
        total_params = num_layers * 2
        x0 = np.zeros(total_params, dtype=np.float64)
        for layer in range(num_layers):
            x0[layer*2:layer*2+2] = np.random.normal(-1, 1, 2)

        def loss_wrapper(p):
            return torch_wrapper_degenerate(p, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers)

        result = minimize(
            lambda p: loss_wrapper(p)[0],
            x0,
            method='L-BFGS-B',
            jac=lambda p: loss_wrapper(p)[1],
            options={'maxiter': maxiter, 'ftol': 1e-12, 'gtol': 1e-10},
            bounds=[(-3, 3)] * total_params
        )
        final_params = result.x
        final_loss = result.fun

        params_list = []
        for layer in range(num_layers):
            re_part = final_params[layer*2]
            im_part = final_params[layer*2+1]
            params_list.append([re_part, im_part])
        psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
        psi_numpy = psi.cpu().detach().numpy()
        norm = np.vdot(psi_numpy, psi_numpy)

        if sp.issparse(H):
            H_psi = H.dot(psi_numpy)
        else:
            H_psi = H @ psi_numpy
        energy = np.real(np.vdot(psi_numpy, H_psi) / norm)

        overlap1 = np.abs(np.vdot(psi_numpy, exact_states[1]['state']))**2 / norm
        overlap2 = np.abs(np.vdot(psi_numpy, exact_states[2]['state']))**2 / norm
        fidelity = overlap1 + overlap2

        print(f"  Loss = {final_loss:.8f}, Energy = {energy:.8f}, Fidelity (ex1+ex2) = {fidelity:.8f}")

        if energy < best_energy:
            best_energy = energy
            best_params = final_params
            best_psi = psi_numpy
            best_fidelity = fidelity

    return best_energy, best_params, best_psi, exact_states, best_fidelity

def ground_loss_func_excited(params, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers):
    params_per_layer = 2
    total_layers = len(params) // params_per_layer
    params_list = []
    for layer in range(total_layers):
        start_idx = layer * params_per_layer
        re_part = params[start_idx]
        im_part = params[start_idx + 1]
        params_list.append([re_part, im_part])
    psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
    H_psi = H_tensor @ psi
    energy = torch.real(torch.vdot(psi, H_psi))
    norm2 = torch.real(torch.vdot(psi, psi))
    return energy / norm2

def torch_wrapper_ground(params_numpy, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers):
    params = torch.tensor(params_numpy, dtype=torch.float64, requires_grad=True)
    loss = ground_loss_func_excited(params, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers)
    loss.backward()
    loss_value = loss.item()
    grad_value = params.grad.numpy().astype(np.float64)
    return loss_value, grad_value

def optimize_ground_state_for_excited(L, delta, num_layers, num_attempts, maxiter, device="cpu"):
    """Find the best Bethe approximation to the XXZ ground state (used as reference for penalty)"""
    print(f"\n--- Ground state fitting (for later penalty) ---")
    H = xxz_hamiltonian(L, delta=delta)
    I2, Sx, Sy, Sz, Sp, Sm = local_ops(dtype=torch.complex128, device=device)
    perms, inv_perms = precompute_perms(L)
    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    H_tensor = torch.tensor(H_dense, dtype=torch.complex128, device=device)

    best_energy = float('inf')
    best_params = None
    best_psi = None

    for attempt in range(num_attempts):
        print(f"  Attempt {attempt+1}/{num_attempts}:")
        total_params = num_layers * 2
        x0 = np.zeros(total_params, dtype=np.float64)
        for layer in range(num_layers):
            x0[layer*2:layer*2+2] = np.random.normal(-1, 1, 2)

        def loss_wrapper(p):
            return torch_wrapper_ground(p, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers)

        try:
            result = minimize(
                lambda p: loss_wrapper(p)[0],
                x0,
                method='BFGS',
                jac=lambda p: loss_wrapper(p)[1],
                options={'maxiter': maxiter, 'ftol': 1e-10, 'gtol': 1e-8}
            )
            final_params = result.x
            final_loss = result.fun

            params_list = []
            for layer in range(num_layers):
                start_idx = layer * 2
                re_part = final_params[start_idx]
                im_part = final_params[start_idx+1]
                params_list.append([re_part, im_part])
            psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
            psi_numpy = psi.cpu().detach().numpy()
            norm = np.vdot(psi_numpy, psi_numpy)
            if sp.issparse(H):
                H_psi = H.dot(psi_numpy)
            else:
                H_psi = H @ psi_numpy
            energy = np.real(np.vdot(psi_numpy, H_psi) / norm)
            print(f"    Loss = {final_loss:.8f}, Energy = {energy:.8f}")
            if energy < best_energy:
                best_energy = energy
                best_params = final_params
                best_psi = psi_numpy
        except Exception as e:
            print(f"    Failed: {e}")
            continue
    return best_energy, best_params, best_psi

def excited_loss_func(params, H_tensor, ground_psi_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms,
                      num_layers, beta_penalty):
    params_per_layer = 2
    total_layers = len(params) // params_per_layer
    params_list = []
    for layer in range(total_layers):
        start_idx = layer * params_per_layer
        re_part = params[start_idx]
        im_part = params[start_idx + 1]
        params_list.append([re_part, im_part])

    psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
    H_psi = H_tensor @ psi
    energy = torch.real(torch.vdot(psi, H_psi))
    norm2 = torch.real(torch.vdot(psi, psi))
    energy_expectation = energy / norm2

    overlap = torch.abs(torch.vdot(psi, ground_psi_tensor))
    overlap_sq = (overlap ** 2) / norm2

    total_loss = energy_expectation + beta_penalty * overlap_sq
    return total_loss

def torch_wrapper_excited(params_numpy, H_tensor, ground_psi_tensor, L, I2, Sz, Sp, Sm,
                          perms, inv_perms, num_layers, beta_penalty):
    params = torch.tensor(params_numpy, dtype=torch.float64, requires_grad=True)
    loss = excited_loss_func(params, H_tensor, ground_psi_tensor, L, I2, Sz, Sp, Sm,
                             perms, inv_perms, num_layers, beta_penalty)
    loss.backward()
    loss_value = loss.item()
    grad_value = params.grad.numpy().astype(np.float64)
    return loss_value, grad_value

def optimize_excited_state_penalty(L, delta, num_layers, num_attempts_ground, num_attempts_excited,
                                   maxiter, device="cpu"):
    """For XXZ model: first excited state via penalty method (BFGS)"""
    print(f"\n=== Excited state fitting (XXZ model with penalty) ===")
    print(f"L={L}, Δ={delta}, layers={num_layers}")
    H = xxz_hamiltonian(L, delta=delta)
    exact_states = exact_diagonalization(H, num_states=3)
    ground_energy_exact = exact_states[0]['energy']
    excited1_energy_exact = exact_states[1]['energy']
    print(f"Exact ground energy: {ground_energy_exact:.8f}")
    print(f"Exact first excited energy: {excited1_energy_exact:.8f}")

    ground_energy_bethe, ground_params, ground_psi = optimize_ground_state_for_excited(
        L, delta, num_layers, num_attempts_ground, maxiter, device
    )
    if ground_psi is None:
        print("Ground state fitting failed.")
        return None, None, None, exact_states, 0.0

    I2, Sx, Sy, Sz, Sp, Sm = local_ops(dtype=torch.complex128, device=device)
    perms, inv_perms = precompute_perms(L)
    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    H_tensor = torch.tensor(H_dense, dtype=torch.complex128, device=device)
    ground_psi_tensor = torch.tensor(ground_psi, dtype=torch.complex128, device=device)

    exact_ground_tensor = torch.tensor(exact_states[0]['state'], dtype=torch.complex128, device=device)
    exact_excited1_tensor = torch.tensor(exact_states[1]['state'], dtype=torch.complex128, device=device)

    best_energy = float('inf')
    best_params = None
    best_psi = None
    best_fidelity_exact = 0.0
    best_overlap_ground_fit = 1.0

    beta_penalty_values = [5, 10, 50]

    for beta_penalty in beta_penalty_values:
        print(f"\nPenalty coefficient β = {beta_penalty:.1f}:")
        for attempt in range(num_attempts_excited):
            print(f"  Attempt {attempt+1}/{num_attempts_excited}:")
            total_params = num_layers * 2
            x0 = np.zeros(total_params, dtype=np.float64)
            for layer in range(num_layers):
                x0[layer*2:layer*2+2] = np.random.normal(-1, 1, 2)

            def loss_wrapper(p):
                return torch_wrapper_excited(
                    p, H_tensor, ground_psi_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms,
                    num_layers, beta_penalty
                )

            try:
                result = minimize(
                    lambda p: loss_wrapper(p)[0],
                    x0,
                    method='BFGS',
                    jac=lambda p: loss_wrapper(p)[1],
                    options={'maxiter': maxiter, 'ftol': 1e-10, 'gtol': 1e-8}
                )
                final_params = result.x
                final_loss = result.fun

                params_list = []
                for layer in range(num_layers):
                    start_idx = layer * 2
                    re_part = final_params[start_idx]
                    im_part = final_params[start_idx+1]
                    params_list.append([re_part, im_part])
                psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
                psi_numpy = psi.cpu().detach().numpy()
                norm = np.vdot(psi_numpy, psi_numpy)

                if sp.issparse(H):
                    H_psi = H.dot(psi_numpy)
                else:
                    H_psi = H @ psi_numpy
                energy = np.real(np.vdot(psi_numpy, H_psi) / norm)

                fidelity_exact = np.abs(np.vdot(psi_numpy, exact_excited1_tensor.cpu().numpy()))**2 / norm
                overlap_ground_fit = np.abs(np.vdot(psi_numpy, ground_psi))**2 / norm

                print(f"    Loss = {final_loss:.8f}, Energy = {energy:.8f}, Fidelity = {fidelity_exact:.8f}, Overlap(ground) = {overlap_ground_fit:.8f}")
                if energy < best_energy and overlap_ground_fit < 0.1:
                    best_energy = energy
                    best_params = final_params
                    best_psi = psi_numpy
                    best_fidelity_exact = fidelity_exact
                    best_overlap_ground_fit = overlap_ground_fit
            except Exception as e:
                print(f"    Failed: {e}")
                continue

    return best_energy, best_params, best_psi, exact_states, best_fidelity_exact

if __name__ == "__main__":
    model = "XXZ"          
    L = 4
    num_layers = L // 2
    num_attempts = 10       
    device = "cuda" if torch.cuda.is_available() else "cpu"
    maxiter = 20000

    if model == "XXZ":
        delta = 1.1
        num_attempts_ground = 10
        num_attempts_excited = 20
        start_time = time.time()
        best_energy, best_params, best_psi, exact_states, best_fidelity = optimize_excited_state_penalty(
            L=L,
            delta=delta,
            num_layers=num_layers,
            num_attempts_ground=num_attempts_ground,
            num_attempts_excited=num_attempts_excited,
            maxiter=maxiter,
            device=device
        )
        elapsed = time.time() - start_time

        if best_psi is not None:
            print("\n=== Final Results (XXZ excited state) ===")
            print(f"Bethe ansatz first excited energy: {best_energy:.8f}")
            print(f"Exact first excited energy:         {exact_states[1]['energy']:.8f}")
            print(f"Energy error:                       {abs(best_energy - exact_states[1]['energy']):.8f}")
            print(f"Fidelity to exact first excited:    {best_fidelity:.8f}")
            overlap_ground = np.abs(np.vdot(best_psi, exact_states[0]['state']))**2 / np.vdot(best_psi, best_psi)
            print(f"Overlap with exact ground state:    {overlap_ground:.8e}")
            print("\nOptimized Bethe parameters u (per layer):")
            for layer in range(num_layers):
                re = best_params[layer*2]
                im = best_params[layer*2+1]
                print(f"  Layer {layer+1}: {re:.6f} + {im:.6f}i")
            print(f"\nTotal time: {elapsed:.2f} seconds")

    elif model == "PeriodicField":

        delta = 1.0
        Q = np.pi
        h = 0.85
        num_layers = L // 2 - 1  
        start_time = time.time()
        best_energy, best_params, best_psi, exact_states, best_fidelity = optimize_degenerate_ground_state(
            model=model,
            L=L,
            num_layers=num_layers,
            num_attempts=num_attempts,
            maxiter=maxiter,
            device=device,
            delta=delta,
            Q=Q,
            h=h
        )
        elapsed = time.time() - start_time

        if best_psi is not None:
            print("\n=== Final Results (PeriodicField degenerate ground state) ===")
            print(f"Bethe ansatz ground energy: {best_energy:.8f}")
            print(f"Exact ground energy:         {exact_states[0]['energy']:.8f}")
            print(f"Energy error:                {abs(best_energy - exact_states[0]['energy']):.8f}")
            print(f"Fidelity (overlap with ex1+ex2): {best_fidelity:.8f}")
            overlap_ex1 = np.abs(np.vdot(best_psi, exact_states[1]['state']))**2 / np.vdot(best_psi, best_psi)
            overlap_ex2 = np.abs(np.vdot(best_psi, exact_states[2]['state']))**2 / np.vdot(best_psi, best_psi)
            print(f"Overlap^2 with first excited:  {overlap_ex1:.8f}")
            print(f"Overlap^2 with second excited: {overlap_ex2:.8f}")
            print("\nOptimized Bethe parameters u (per layer):")
            for layer in range(num_layers):
                re = best_params[layer*2]
                im = best_params[layer*2+1]
                print(f"  Layer {layer+1}: {re:.6f} + {im:.6f}i")
            print(f"\nTotal time: {elapsed:.2f} seconds")